# Data Preprocess

In [ ]:
import os
import time
import uuid
from qdrant_client import QdrantClient
from utils.config import config
from utils.embedding import query_embedding
from utils.chunking import get_splitter
from utils.qdrant import create_collection, insert_single_point
import logging
logger = logging.getLogger(__name__)


# Load data md
def scan_md(dir_path) -> list:
    if not os.path.exists(dir_path):
        logger.error(f"Directory {dir_path} does not exist.")
        return []

    md_files = []
    for file in os.listdir(dir_path):
        if file.endswith(".md"):
            md_files.append(file)
    logger.info(f"Found {len(md_files)} markdown files.")
    return md_files

# Chunking & Embedding & Store
text_splitter = get_splitter()

client = QdrantClient(host=config.qdrant.host, port=config.qdrant.port)
if not client.collection_exists(config.qdrant.collection_name):
    create_collection(client, config.qdrant.collection_name, vector_size=1024)


md_files = scan_md("./data/md/")
for md_file in md_files:
    text = ""
    with open(f"./data/md/{md_file}", "r", encoding="utf-8") as f:
        text = f.read()
        chunks = text_splitter.split_text(text)
        print(f"File: {md_file}, Original Length: {len(text)}, Chunks: {len(chunks)}")

        for chunk_idx, chunk in enumerate(chunks):
            time.sleep(20)  # 避免过快请求 API 导致问题
            q_embedding = query_embedding(chunk)  # 生成向量并存储到 Qdrant
            point_id = str(uuid.uuid4())  # 生成唯一 UUID
            insert_single_point(client, config.qdrant.collection_name, point_id, q_embedding, {"text": chunk, "original_id": f"{md_file}_{chunk_idx}"})

client.close()

# RAG

In [ ]:
from qdrant_client import QdrantClient
from utils.config import config
from utils.embedding import query_embedding

def query_enhance(query):
    # Todo: MQE
    # Todo: HyDE
    # Todo: 意图识别
    return query

def query_retrieval(q_embedding):
    client = QdrantClient(host=config.qdrant.host, port=config.qdrant.port)

    results = client.query_points(
        collection_name=config.qdrant.collection_name,
        query=q_embedding,
        limit=2
    )
    client.close()
    return results

def context_augmentation(user_query, results):
    prompt = ""
    for idx, result in enumerate(results.points):
        text = result.payload["text"]
        prompt += f"参考信息{idx+1}: {text}\n"
    prompt += f"用户问题：{user_query}\n请基于以上参考信息回答用户问题，要求内容准确且详细。"
    return prompt


# Chat

In [ ]:
import openai
from utils.config import config

user_query = "请介绍一下顺丰发展情况"
query_enhanced = query_enhance(user_query)
q_embedding = query_embedding(query_enhanced)
results = query_retrieval(q_embedding)
prompt = context_augmentation(user_query, results)

client = openai.OpenAI(
    base_url = config.chat.url,
    api_key = config.chat.token
)

response = client.chat.completions.create(
    model=config.chat.model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    # 设置 reasoning_split=True 将思考内容分离到 reasoning_details 字段
    # extra_body={"reasoning_split": True},
)

# print(f"Thinking:\n{response.choices[0].message.reasoning_details[0]['text']}\n")
print(f"Text:\n{response.choices[0].message.content}\n")

# Test

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from ragas.run_config import RunConfig
from utils.config import config
from utils.chunking import get_splitter

loader = DirectoryLoader(
    path="data/md/",
    glob="**/*.md",
    loader_cls=UnstructuredFileLoader,
    loader_kwargs={
        "mode": "single",
        "strategy": "fast",
        "languages": ["zh"]
    }
)
docs = loader.load()
print("Loaded docs:", len(docs))

splitter = get_splitter()
docs = splitter.split_documents(docs)
print("Chunks:", len(docs))

docs = [d for d in docs if len(d.page_content) > 50]
print("Filtered docs:", len(docs))

client = OpenAI(
    api_key="sk-jgxoeujwhevcvthuyqeqrplalzgzsmgavhctoggxspvyiakw",
    base_url="https://api.siliconflow.cn/v1"
)
llm = llm_factory(
    model="deepseek-ai/DeepSeek-V3",
    client=client,
    temperature=0
)
embedding_model = OpenAIEmbeddings(
    model=config.embed.model,
    client=client
)
generator = TestsetGenerator(
    llm=llm,
    embedding_model=embedding_model
)
run_config = RunConfig(
    max_workers=1
)
dataset = generator.generate_with_langchain_docs(
    docs,
    testset_size=2,
    run_config=run_config,
    with_debugging_logs=True,
    raise_exceptions=False
)

df = dataset.to_pandas()
print(df.head())

df.to_csv("data/rag_testset.csv", index=False)
print("Saved to rag_testset.csv")